# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `book/chapters/ch05_data_preprocessing.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 데이터를 무조건 깨끗하게 만드는 것이 아니라, 분석 결과를 신뢰할 수 있도록 데이터의 상태를 확인하고 처리 기준을 남기는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/`에 그대로 둡니다.
- 전처리 결과는 `data/processed/`에 별도로 저장합니다.
- 전처리 요약 보고서는 `reports/ch05_preprocessing_summary.md`에 저장합니다.
- 코드가 실행되었다고 끝이 아니라, 처리 기준을 설명할 수 있어야 합니다.


## 1. 왜 전처리가 필요한가

현실의 데이터는 처음부터 분석하기 좋은 형태로 주어지지 않습니다.

예를 들어 다음과 같은 문제가 있을 수 있습니다.

| 문제 유형 | 예시 | 확인할 질문 |
|---|---|---|
| 결측치 | 나이가 비어 있는 고객 | 비어 있는 이유가 무엇인가? 삭제해도 되는가? |
| 중복 | 같은 고객 ID가 여러 번 등장 | 자연스러운 중복인가, 데이터 오류인가? |
| 타입 오류 | 가격이 문자열로 저장됨 | 계산 가능한 숫자형으로 바꿀 수 있는가? |
| 날짜 오류 | 주문일이 문자열로 저장됨 | 월별·요일별 분석에 사용할 수 있는가? |
| 문자열 표기 차이 | `Seoul`, ` Seoul`, `SEOUL` | 같은 값을 하나의 표기로 통일해야 하는가? |
| 이상값 | 수량이 음수이거나 가격이 0원 | 실제 의미가 있는 값인가, 입력 오류인가? |
| 파생 컬럼 필요 | 수량과 단가만 있고 주문금액이 없음 | 분석에 필요한 새 컬럼을 만들 수 있는가? |


## 2. 원본 데이터와 전처리 데이터 분리하기

전처리에서 가장 중요한 원칙은 원본 데이터를 직접 덮어쓰지 않는 것입니다.

```text
data/
├─ raw/
│  ├─ customers.csv
│  ├─ products.csv
│  ├─ orders.csv
│  └─ order_items.csv
└─ processed/
   ├─ customers_clean.csv
   ├─ products_clean.csv
   ├─ orders_clean.csv
   └─ order_items_clean.csv
```

원본과 전처리 결과를 분리하면 실수했을 때 원본으로 돌아갈 수 있고, 전처리 전후 차이를 비교할 수 있습니다.


## 3. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [36]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: c:\Users\ojh23\Documents\projects\llm-data-analysis-course\notebooks
프로젝트 루트: c:\Users\ojh23\Documents\projects\llm-data-analysis-course
원본 데이터 폴더: c:\Users\ojh23\Documents\projects\llm-data-analysis-course\data\raw
전처리 데이터 폴더: c:\Users\ojh23\Documents\projects\llm-data-analysis-course\data\processed
보고서 폴더: c:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports


## 4. 원본 데이터 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보


In [37]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')


원본 데이터 불러오기 완료


## 5. 전처리 전 데이터 크기 기록하기

전처리 전 데이터 크기를 기록해 두면 나중에 어떤 데이터가 얼마나 바뀌었는지 비교할 수 있습니다.


In [38]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes


,dataset,rows,columns
0,customers,150,6
1,products,100,4
2,orders,300,5
3,order_items,764,5


In [39]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (764, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



## 6. 원본을 복사해서 전처리 시작하기

전처리 과정에서는 원본 DataFrame을 직접 수정하지 않고 복사본을 사용합니다.


In [40]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')


전처리용 복사본 생성 완료


## 7. 결측치 확인하기

결측치를 처리하기 전에 먼저 어디에 얼마나 비어 있는 값이 있는지 확인합니다. 개수와 비율을 함께 보면 데이터 크기에 따른 차이를 이해하기 쉽습니다.


In [41]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())



[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [42]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

missing_summary(customers_clean)


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 8. 결측치 처리하기

결측치 처리는 컬럼의 의미에 따라 달라집니다.

- 나이처럼 숫자형 컬럼은 중앙값으로 대체할 수 있습니다.
- 도시처럼 범주형 컬럼은 `Unknown`으로 남겨 분석에서 구분할 수 있습니다.

실제 업무에서는 결측치가 왜 생겼는지 먼저 확인해야 합니다.


In [43]:
if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    age_median = customers_clean['age'].median()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

missing_summary(customers_clean)


age 중앙값: 40.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 9. 중복 확인하기

중복은 맥락에 따라 의미가 다릅니다. `customers`의 `customer_id` 중복은 오류일 가능성이 크지만, `order_items`의 `order_id` 반복은 한 주문에 여러 상품이 들어갈 수 있으므로 자연스럽습니다.


In [44]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())


customers 전체 행 중복 수: 0
products 전체 행 중복 수: 0
orders 전체 행 중복 수: 0
order_items 전체 행 중복 수: 0


In [45]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')


customers customer_id 중복 수: 0
products product_id 중복 수: 0
orders order_id 중복 수: 0
order_items order_item_id 중복 수: 0


In [46]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')


완전 중복 행 제거 완료


## 10. 문자열 표기 정리하기

문자열 데이터에는 앞뒤 공백이 숨어 있을 수 있습니다. `Seoul`과 ` Seoul `은 눈으로는 비슷하지만 pandas에서는 서로 다른 값입니다.

결측치를 문자열 `nan`으로 바꾸지 않도록 주의하면서 문자열 컬럼의 앞뒤 공백을 제거합니다.


In [47]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')


문자열 앞뒤 공백 제거 완료


In [48]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 11. 주문 상태값 통일하기

주문 상태값이 `complete`, `Complete`, `COMPLETED`, `완료`처럼 섞여 있으면 같은 상태를 서로 다른 값으로 집계할 수 있습니다. 대표 표기로 통일합니다.


In [49]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 12. 날짜 컬럼 변환하기

날짜처럼 보이는 값도 실제로는 문자열일 수 있습니다. 월별 주문 분석이나 요일별 분석을 하려면 날짜형으로 변환해야 합니다.

`errors="coerce"`는 변환할 수 없는 값을 `NaT`로 바꿉니다. 그래서 변환 후 실패 건수를 반드시 확인해야 합니다.


In [50]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()


order_date 변환 실패: 0
주문 시작일: 2025-07-09 00:00:00
주문 종료일: 2026-07-08 00:00:00


,order_date,order_month,order_dayofweek
0,2026-05-07,2026-05,Thursday
1,2025-07-23,2025-07,Wednesday
2,2025-11-19,2025-11,Wednesday
3,2026-01-30,2026-01,Friday
4,2025-12-21,2025-12,Sunday


In [51]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())


signup_date 변환 실패: 0


,customer_id,signup_date
0,1,2024-06-19
1,2,2025-11-02
2,3,2024-06-12
3,4,2026-04-13
4,5,2024-09-13


## 13. 숫자형 컬럼 변환하기

가격, 수량, 단가는 숫자처럼 보여도 문자열일 수 있습니다. 특히 `10,000`처럼 쉼표가 들어간 값은 바로 계산하기 어렵습니다.


In [52]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())


price 변환 실패: 0
quantity 변환 실패: 0
unit_price 변환 실패: 0


## 14. 이상값 후보 확인하기

이상값은 일반적인 범위를 벗어난 값입니다. 하지만 이상값이 항상 오류는 아닙니다. 0원 상품은 이벤트 상품일 수 있고, 음수 수량은 반품을 의미할 수도 있습니다. 먼저 확인하고 기준을 정해야 합니다.


In [53]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())


count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

,quantity,unit_price
count,764.000000,764.000000
mean,3.053665,108561.518325
std,1.410873,56996.770604
min,1.000000,5000.000000
25%,2.000000,62000.000000
50%,3.000000,111000.000000
75%,4.000000,161250.000000
max,5.000000,200000.000000


In [54]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))


price <= 0: 0
quantity <= 0: 0
unit_price <= 0: 0


### 실습용 이상값 처리 기준

이번 실습에서는 정상 주문 분석을 위해 다음 기준을 적용합니다.

```text
- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외
```

실제 업무에서는 이 기준을 적용하기 전에 반드시 원본 시스템이나 담당자 확인이 필요합니다.


In [55]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)


이상값 처리 후 products: (100, 4)
이상값 처리 후 order_items: (764, 5)


## 15. 파생 컬럼 만들기

전처리된 수량과 단가를 사용하면 주문 상세 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [56]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,3,102000,306000
1,5,25000,125000
2,3,142000,426000
3,3,193000,579000
4,4,189000,756000


In [57]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())


전처리 후 주문 상세 금액 합계: 255610000


## 16. 파일 간 관계 다시 확인하기

전처리 과정에서 일부 행을 삭제하면 파일 간 관계가 깨질 수 있습니다. 예를 들어 `products`에서 일부 상품을 제외했는데 `order_items`에는 그 상품 ID가 남아 있을 수 있습니다.


In [58]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


## 17. 전처리 전후 비교하기

전처리 전후 데이터 크기를 비교합니다. 행 수가 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 하고, 열 수가 늘었다면 어떤 파생 컬럼이 추가되었는지 설명할 수 있어야 합니다.


In [59]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,products,100,4,100,4
2,orders,300,5,300,7
3,order_items,764,5,764,6


## 18. 전처리 결과 저장하기

전처리된 데이터는 `data/processed` 폴더에 저장합니다. Excel에서 한글 CSV를 바로 열 수 있도록 `utf-8-sig` 인코딩을 사용합니다.


In [60]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

list(PROCESSED_DIR.glob('*_clean.csv'))


[WindowsPath('c:/Users/ojh23/Documents/projects/llm-data-analysis-course/data/processed/customers_clean.csv'),
 WindowsPath('c:/Users/ojh23/Documents/projects/llm-data-analysis-course/data/processed/orders_clean.csv'),
 WindowsPath('c:/Users/ojh23/Documents/projects/llm-data-analysis-course/data/processed/order_items_clean.csv'),
 WindowsPath('c:/Users/ojh23/Documents/projects/llm-data-analysis-course/data/processed/products_clean.csv')]

## 19. 전처리 요약 보고서 저장하기

전처리 기준과 결과를 Markdown 파일로 남겨 두면 이후 보고서 작성이 쉬워집니다.


In [61]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)


요약 보고서 저장 완료: c:\Users\ojh23\Documents\projects\llm-data-analysis-course\reports\ch05_preprocessing_summary.md


## 20. 전처리 과정을 함수로 정리하기

전처리 코드를 한 번만 실행하고 끝내면 재사용하기 어렵습니다. 같은 데이터를 다시 받거나 처리 기준을 조금 바꾸어야 할 때는 함수로 정리된 코드가 훨씬 유용합니다.


In [62]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    return result.drop_duplicates()


In [63]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)


(150, 6)
(100, 4)
(300, 7)
(764, 6)


## 21. 소스 모듈 사용하기

위에서 직접 작성한 전처리 함수는 `src/preprocessing.py`에도 정리되어 있습니다. 반복 작업이나 실제 프로젝트에서는 노트북 안에만 코드를 두기보다 소스 모듈로 분리하는 것이 좋습니다.


In [64]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)

module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,order_items,764,5,764,6
2,orders,300,5,300,7
3,products,100,4,100,4


In [65]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)


,dataset,row_duplicate_count,key_column,key_duplicate_count
0,customers,0,customer_id,0
1,products,0,product_id,0
2,orders,0,order_id,0
3,order_items,0,order_item_id,0


In [66]:
module_relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,0
1,order_items.order_id exists in orders.order_id,0
2,order_items.product_id exists in products.prod...,0


## 22. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 전처리 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/preprocess_data.py
```

이 스크립트는 전처리 결과 CSV와 요약 보고서를 자동으로 저장합니다.


## 23. LLM과 함께 전처리 코드를 검토하기

LLM에게 전처리 코드를 요청할 때는 실제 고객명, 이메일, 주문 내역을 그대로 입력하지 않는 것이 좋습니다. 컬럼명, 데이터 타입, 결측치 개수, 중복 여부, 처리 목적처럼 구조화된 정보만 제공하세요.

```text
다음 전처리 코드가 안전한지 검토해 주세요.

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지
```


### LLM 제안 검증

LLM은 원본 DataFrame을 직접 수정하는 문제, 평균으로 결측치를 채우기 전 분포 확인 필요성, 날짜 변환 실패 확인, 중복 기준 분리, `line_total` 계산 전 숫자형 변환·범위 점검을 제안했다.

### 실제 데이터 확인 및 판단

- 원본 보존: Notebook에서 `customers_clean`, `products_clean`, `orders_clean`, `order_items_clean` 복사본을 만들어 전처리했으므로 **채택**했다.
- age 결측치: 실제 결측치는 0건이므로 평균 대체는 실행하지 않았다. 결측치가 생길 경우에만 분포와 업무 기준을 확인한 뒤 처리해야 하므로 **수정 후 채택**했다.
- 날짜 변환: `pd.to_datetime(..., errors="coerce")`로 변환한 뒤 실패 건수를 확인했고, `order_date` 변환 실패는 0건이었다. 따라서 **채택**했다.
- 중복 제거: 전체 행 중복과 주요 ID 중복을 구분해 확인했고 모두 0건이었다. 단순 `drop_duplicates()`만 사용하는 방식은 기준이 불명확하므로 **수정 후 채택**했다.
- line_total: `quantity`, `unit_price`를 숫자형으로 변환한 뒤 변환 실패와 0 이하 값을 확인했다. 모두 0건이어서 `line_total = quantity × unit_price`를 계산했다. 따라서 **채택**했다.

### 최종 판단

LLM의 제안은 안전한 전처리 기준을 점검하는 데 활용했다. 다만 실제 처리 여부는 LLM 답변이 아니라 실행 결과를 기준으로 결정했다.

## 24. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
2. `customers_clean`에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
3. `orders_clean`에서 월별 주문 건수를 계산하세요.
4. `order_items_clean`에서 `line_total`이 큰 순서대로 상위 10개를 확인하세요.
5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.
6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.


In [71]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
# hint: missing_summary(df).reset_index()를 활용해 보세요.

# 각 데이터셋의 컬럼별 결측치 개수·비율 통합
missing_tables = []

for dataset_name, df in {
    "customers": customers_clean,
    "products": products_clean,
    "orders": orders_clean,
    "order_items": order_items_clean,
}.items():
    summary = missing_summary(df).reset_index()
    summary = summary.rename(columns={"index": "column"})
    summary.insert(0, "dataset", dataset_name)
    missing_tables.append(summary)

task_missing_summary = pd.concat(missing_tables, ignore_index=True)

print("[과제 1] 데이터셋별 결측치 개수와 비율")
display(task_missing_summary)

[과제 1] 데이터셋별 결측치 개수와 비율


,dataset,column,missing_count,missing_ratio
0,customers,customer_id,0,0.0
1,customers,name,0,0.0
2,customers,gender,0,0.0
3,customers,age,0,0.0
4,customers,city,0,0.0
5,customers,signup_date,0,0.0
6,products,product_id,0,0.0
7,products,product_name,0,0.0
8,products,category,0,0.0
9,products,price,0,0.0


In [72]:
# 과제 2. customers_clean에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.

# 나이 이상값 후보 확인
invalid_age = customers_clean.loc[
    (customers_clean["age"] < 18) | (customers_clean["age"] > 100),
    ["customer_id", "age"]
]

print("[과제 2] 18세 미만 또는 100세 초과 고객 수:", len(invalid_age))
display(invalid_age)

[과제 2] 18세 미만 또는 100세 초과 고객 수: 0


,customer_id,age


In [73]:
# 과제 3. orders_clean에서 월별 주문 건수를 계산하세요.

# 월별 주문 건수: 주문 ID의 고유 개수 기준
monthly_order_count = (
    orders_clean
    .groupby("order_month", as_index=False)
    .agg(order_count=("order_id", "nunique"))
    .sort_values("order_month")
)

print("[과제 3] 월별 주문 건수")
display(monthly_order_count)

[과제 3] 월별 주문 건수


,order_month,order_count
0,2025-07,15
1,2025-08,24
2,2025-09,23
3,2025-10,39
4,2025-11,23
5,2025-12,26
6,2026-01,29
7,2026-02,29
8,2026-03,21
9,2026-04,31


In [74]:
# 과제 4. order_items_clean에서 line_total이 큰 순서대로 상위 10개를 확인하세요.

# 주문 상세 금액 상위 10건
top10_line_total = (
    order_items_clean[
        ["order_item_id", "order_id", "product_id", "quantity", "unit_price", "line_total"]
    ]
    .sort_values("line_total", ascending=False)
    .head(10)
)

print("[과제 4] line_total 상위 10개")
display(top10_line_total)

[과제 4] line_total 상위 10개


,order_item_id,order_id,product_id,quantity,unit_price,line_total
22,23,10,99,5,200000,1000000
85,86,39,99,5,200000,1000000
183,184,76,99,5,200000,1000000
493,494,190,70,5,198000,990000
78,79,35,70,5,198000,990000
631,632,247,58,5,197000,985000
604,605,235,58,5,197000,985000
113,114,51,58,5,197000,985000
569,570,221,58,5,197000,985000
45,46,21,58,5,197000,985000


### 과제 5. 전처리 전후 행 수 변화

customers, products, orders, order_items 모두 전처리 전후 행 수가 줄어들지 않았다.  
이번 데이터에서는 결측치, 전체 행 중복, 주요 ID 중복, 숫자형 변환 실패, 0 이하 값이 모두 0건이었기 때문이다.  
다만 이는 원본 데이터에 문제가 없다는 뜻이 아니라, 이번에 적용한 전처리 기준에서 제거 대상이 없었다는 의미이다.

### 과제 6. LLM 결측치 처리 기준 검토 프롬프트

다음은 온라인 쇼핑몰 데이터의 결측치 점검 결과입니다.

- customers, products, orders, order_items의 컬럼별 결측치 개수와 비율을 확인했습니다.
- 현재 데이터에서는 주요 컬럼의 결측치가 0건입니다.
- 향후 결측치가 발견될 경우, 단순 삭제·평균·중앙값·최빈값 대체 중 어떤 기준을 적용할지 검토하고 싶습니다.

다음 기준으로 결측치 처리 방안을 제안해 주세요.

1. 컬럼의 역할과 자료형에 따라 어떤 처리 방법이 적절한지
2. 삭제와 대체를 선택하기 전에 확인할 사항
3. 고객 ID, 주문 ID 같은 식별자 결측치를 어떻게 다뤄야 하는지
4. 숫자형·범주형·날짜형 컬럼별 주의점
5. 실제 데이터에 없는 컬럼이나 업무 규칙을 가정하지 말 것

# Chapter 05 제출 답안

## 0. 제출 정보

- 이름: 
- GitHub ID: OBHAR
- 작성일: 2026-10-10
- 최종 제출 URL: GitHub 업로드 후 작성

## 1. 전처리 전 상태 기록

- 각 데이터 shape: customers (150, 6), products (100, 4), orders (300, 5), order_items (764, 5)
- 결측치: 네 데이터셋의 모든 컬럼에서 0건이었다.
- 전체 중복: 네 데이터셋 모두 0건이었다.
- 주요 ID 중복: customer_id, product_id, order_id, order_item_id의 중복은 모두 0건이었다.
- 타입 문제 후보: order_date와 signup_date는 날짜형 변환이 필요하고, price·quantity·unit_price는 숫자형 여부를 확인해야 했다.

![전처리 전 상태](images/step01_before.png)

### 결과 관찰

원본 데이터의 행 수와 주요 ID의 중복, 결측치를 먼저 확인했다. 이번 데이터에서는 즉시 삭제할 결측·중복 행은 확인되지 않았다.

### 나의 해석과 판단

행을 제거하기 전에 날짜·숫자형 변환과 파일 간 키 관계를 우선 점검했다. 결측치와 중복이 0건이더라도 자료형 오류나 참조 무결성 문제는 남을 수 있기 때문이다.

### 업무·분석적 의미

전처리 전 기준선을 남겨 두면 이후 행 수·값의 변화가 의도한 처리인지 확인할 수 있다.

### 한계와 추가 확인 사항

현재 데이터에서 0건이라는 결과만으로 미래 수집 데이터도 같은 품질이라고 단정할 수 없다.

## 2. 문자열·타입·결측 처리

- 적용한 문자열 정리: 주요 문자열 컬럼의 앞뒤 공백을 제거하고 order_status 표기를 통일했다.
- 숫자 변환 대상과 실패 건수: price, quantity, unit_price를 숫자형으로 변환했으며 변환 실패는 모두 0건이었다.
- 날짜 변환 대상과 실패 건수: order_date, signup_date를 `errors="coerce"`로 변환했고 실패는 0건이었다.
- 결측 처리 기준: age는 중앙값, city는 Unknown으로 처리할 수 있도록 기준을 두었으나 실제 결측치는 0건이라 대체는 발생하지 않았다.

![타입과 결측 처리](images/step02_cleaning.png)

### 결과 관찰

자료형 변환 실패와 결측치는 모두 0건이었다. 날짜 변환 후 orders에는 order_month와 order_dayofweek 파생 컬럼을 생성했다.

### 나의 해석과 판단

대체 처리를 무조건 적용하지 않고 결측 여부와 변환 실패를 먼저 확인했다. 결측치가 있는 경우에도 숫자형은 분포를 확인한 뒤 중앙값 대체 여부를 결정하고, 범주형은 Unknown 처리로 원래 결측 여부를 구분하는 것이 적절하다.

### 업무·분석적 의미

날짜·숫자형 변환이 확인되어야 월별 집계와 주문 금액 계산을 신뢰할 수 있다.

### 한계와 추가 확인 사항

현재 날짜가 실제 주문 발생일인지, 가격이 정가인지 판매 시점 가격인지 등 업무 정의는 별도 확인이 필요하다.

## 3. 중복·범주 표준화·이상값 후보

- 제거/보류한 중복: 전체 행 중복과 주요 ID 중복이 모두 0건이어서 제거하지 않았다.
- 표준화한 범주값: order_status의 공백·표기 차이를 정리하고 completed, cancelled, refunded 값으로 확인했다.
- 허용값 밖 값: price, quantity, unit_price의 숫자 변환 실패와 0 이하 값은 모두 0건이었다.
- 이상값 후보: 가격·수량·단가의 분포를 확인했으며, 확인된 값만으로 자동 삭제하지 않았다.

![중복 범주 이상값](images/step03_rules.png)

### 결과 관찰

상품 가격과 단가는 5,000~200,000, 수량은 1~5 범위였고 0 이하 값은 없었다.

### 나의 해석과 판단

높거나 낮은 값은 오류일 수 있지만, 고가 상품이나 대량 구매처럼 실제 업무상 가능한 값일 수도 있다. 따라서 단순 기준으로 자동 삭제하지 않고 이상값 후보로 기록했다.

### 업무·분석적 의미

중복·이상값 처리 기준을 남기면 분석 결과의 변화를 재현하고 설명하기 쉬워진다.

### 한계와 추가 확인 사항

가격과 수량의 허용 범위는 현재 데이터 분포가 아니라 상품 정책과 주문 업무 규칙으로 최종 확인해야 한다.

## 4. PK/FK와 파생 컬럼 재검증

- 없는 customer_id: 0건
- 없는 order_id: 0건
- 없는 product_id: 0건
- 생성한 파생 컬럼: orders_clean의 order_month, order_dayofweek와 order_items_clean의 line_total
- line_total 검증: quantity × unit_price로 계산했으며 전체 line_total 합계는 255,610,000이다.

![관계와 파생 컬럼 검증](images/step04_validation.png)

### 결과 관찰

orders→customers, order_items→orders, order_items→products 관계에서 부모 테이블에 없는 키는 모두 0건이었다.

### 나의 해석과 판단

전처리 과정에서 값 변환·행 제거가 발생하면 키 관계가 깨질 수 있으므로, 전처리 후에도 참조 무결성을 다시 확인해야 한다. 이번 데이터에서는 연결 가능한 상태임을 확인했다.

## 5. 전처리 전/후 비교

| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| customers 행 수 | 150 | 150 | 제거 대상 없음 |
| products 행 수 | 100 | 100 | 제거 대상 없음 |
| orders 행 수 | 300 | 300 | 제거 대상 없음, 날짜 파생 컬럼 2개 추가 |
| order_items 행 수 | 764 | 764 | 제거 대상 없음, line_total 1개 추가 |
| 결측·중복·변환 실패 | 0건 | 0건 | 확인 결과 처리 대상 없음 |

![전처리 전후 비교](images/step05_before_after.png)

### 결과 관찰

모든 데이터셋의 행 수는 유지되었고, 분석에 필요한 파생 컬럼만 추가되었다.

### 나의 해석과 판단

행 수가 유지된 이유는 오류가 없어서라기보다 이번에 정한 제거 기준에 해당하는 행이 없었기 때문이다. 처리 전후 행 수와 실패 건수를 함께 비교해 의도하지 않은 손실이 없음을 확인했다.

### 업무·분석적 의미

전처리로 인해 분석 대상이 달라졌는지 기록하면 결과를 해석할 때 모집단 변화 가능성을 고려할 수 있다.

### 한계와 추가 확인 사항

행 수가 같아도 문자열 표준화나 자료형 변환에 따른 값의 의미 변화는 별도로 점검해야 한다.

## 6. 재현 가능한 전처리 확인

- 생성된 clean CSV: customers_clean.csv, products_clean.csv, orders_clean.csv, order_items_clean.csv
- 생성된 요약 보고서: reports/ch05_preprocessing_summary.md
- 재실행 명령: 프로젝트 루트에서 `python -m scripts.preprocess_data`
- 재실행 후 달라진 점: Notebook과 동일하게 행 수, 중복 점검, 키 관계 점검 결과가 재현되었다.

![재실행 결과](images/step06_reproducible.png)

### 나의 해석과 판단

Notebook에서 한 번 성공한 결과는 셀 실행 순서나 메모리 상태에 의존할 수 있다. 스크립트로 같은 결과가 재현되어야 다른 환경에서도 같은 전처리 기준을 적용할 수 있다.

## 7. Chapter 05 최종 판단

### 내가 정한 전처리 원칙 3가지

1. 원본 DataFrame과 raw CSV를 직접 수정하지 않고 clean 복사본에서 처리한다.
2. 결측·중복·변환 실패를 확인한 뒤 처리하며, 자동 삭제·대체의 기준을 기록한다.
3. 파생 컬럼 생성과 전처리 후에는 PK/FK 관계와 처리 전후 결과를 다시 검증한다.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유

결측 나이를 평균으로 즉시 대체하는 처리이다. 나이 분포를 왜곡하거나 결측 자체의 의미를 숨길 수 있으므로, 결측 원인과 분포를 확인한 뒤 업무 기준에 따라 결정해야 한다.

### 다음 EDA에서 특히 주의할 데이터 특성

주문 상태별 범위를 구분해야 한다. completed, cancelled, refunded 주문을 같은 매출로 해석하지 않고 분석 목적에 맞는 포함 기준을 먼저 정해야 한다.

### 현재 전처리의 한계

현재 데이터만으로 주문 취소·환불의 업무적 의미, 가격의 기준 시점, 이상값의 실제 오류 여부는 판단할 수 없다. 다음 EDA에서는 주문 상태와 기간별 분포를 확인하고 필요하면 업무 담당자에게 기준을 확인해야 한다.

## 최종 제출 체크

- [x] raw 원본을 덮어쓰지 않았습니다.
- [x] 처리 전/후 비교를 남겼습니다.
- [x] 처리 기준과 이유를 작성했습니다.
- [x] clean CSV와 재실행 결과를 확인했습니다.
- [x] 개인정보와 Secret이 없습니다.
- [ ] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시되는지 업로드 후 확인합니다.
- [ ] 최종 Notebook 파일 URL을 LMS에 제출합니다.


## 25. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 원본 데이터와 전처리 데이터 분리
- 결측치 개수와 비율 확인
- 나이 결측치 중앙값 대체, 도시 결측치 Unknown 처리
- 중복 행과 주요 ID 중복 점검
- 문자열 앞뒤 공백 제거와 상태값 표기 통일
- 날짜형 변환과 주문 월/요일 파생 컬럼 생성
- 숫자형 변환과 변환 실패 건수 확인
- 이상값 후보 확인과 실습용 처리 기준 적용
- 주문 상세 금액 `line_total` 생성
- 파일 간 키 관계 재확인
- 전처리 결과 CSV와 요약 보고서 저장
- 전처리 함수를 `src/preprocessing.py` 소스 모듈로 분리

다음 장에서는 전처리된 데이터를 바탕으로 탐색적 데이터 분석, 즉 EDA를 수행합니다.
